<a href="https://colab.research.google.com/github/tekpinar/gromacscolab/blob/main/MD_Preparation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#GROMACSCOLAB: Protein Preparation Notebook

🔷 Workflow Overview

This notebook guides you step by step for preparing a protein for molecular dynamics (MD) simulations using GROMACS.

You will:

* Upload and clean a protein structure
* Select chains and simulation parameters
* Assign protonation states
* Solvate the system
* Add ions interactively
* Visualize the final structure

⚙️ Setup:
Install required software (GROMACS, PDB2PQR, and visualization tools).
Run this step once before starting.

In [ ]:
!apt-get update -qq
!apt-get install -y gromacs
!pip install pdb2pqr
!pip install py3Dmol
print("GROMACS and PDB2PQR installations completed.")

# Upload PDB File
Upload your protein structure file (.pdb) to start the workflow.
The file name will be automatically cleaned and formatted.

In [ ]:
from google.colab import files
import os
import re

def upload_pdb():
    uploaded = files.upload()
    pdb_file = list(uploaded.keys())[0]

    safe = re.sub(r'[^A-Za-z0-9_.-]', '_', pdb_file)
    if safe != pdb_file:
        os.rename(pdb_file, safe)
        pdb_file = safe

    base, ext = os.path.splitext(pdb_file)
    if ext.lower() != ".pdb":
        new_name = base + ".pdb"
        os.rename(pdb_file, new_name)
        pdb_file = new_name

    print("PDB file ready:", pdb_file)
    return pdb_file

pdb_file = upload_pdb()
base_name = os.path.splitext(os.path.basename(pdb_file))[0]

print("Loaded file:", pdb_file)

# Select Chains
All chains are selected by default.
You can uncheck any chains you want to exclude.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

chains = set()

with open(pdb_file) as f:
    for line in f:
        if line.startswith(("ATOM", "HETATM")):
            chain_id = line[21]
            if chain_id.strip() == "":
                chains.add("A")
            else:
                chains.add(chain_id)

chain_list = sorted(list(chains))

print(f"🔗 Total {len(chain_list)} chains found:", chain_list)

chain_checkboxes = []
for chain in chain_list:
    cb = widgets.Checkbox(value=True, description=f"Chain {chain}")
    chain_checkboxes.append(cb)

display(*chain_checkboxes)

# Generate PDB from Selected Chains
Creates a new PDB file based on your selected chains.
This file will be used for visualization and GROMACS simulations.

In [ ]:
selected_chains = [
    chain_list[i]
    for i, cb in enumerate(chain_checkboxes)
    if cb.value
]

chain_string = "".join(selected_chains)
file_prefix = f"{base_name}_{chain_string}"

visual_pdb = f"{file_prefix}_visual.pdb"

with open(pdb_file) as infile, open(visual_pdb, "w") as outfile:
    for line in infile:
        if line.startswith(("ATOM","HETATM")):
            cid = line[21].strip() or "A"
            if cid in selected_chains:
                outfile.write(line)
        else:
            outfile.write(line)

print("Visualization PDB:", visual_pdb)

gmx_input_pdb = f"{file_prefix}_gmx.pdb"

with open(visual_pdb) as infile, open(gmx_input_pdb, "w") as outfile:
    for line in infile:
        outfile.write(line)

print("GROMACS input PDB:", gmx_input_pdb)

# Select Simulation Parameters
Choose the solvent, force field, box type, and other simulation parameters.
Default values are recommended, but you can adjust them if needed.

👉 If unsure, you can proceed with the default settings.

In [ ]:
solvent_widget = widgets.Dropdown(
    options=['tip3p','spc','spce','tip4p'],
    value='tip3p',
    description="Solvent:"
)

ff_widget = widgets.Dropdown(
    options=['amber99sb-ildn','charmm36','opls-aa'],
    value='amber99sb-ildn',
    description="Force Field:"
)

buffer_widget = widgets.FloatText(
    value=1.1,
    description="Buffer (nm):"
)

ph_widget = widgets.FloatSlider(
    value=7.4,
    min=4.0,
    max=9.0,
    step=0.1,
    description="pH:"
)

box_type_widget = widgets.Dropdown(
    options=['cubic','dodecahedron','octahedron', 'triclinic'],
    value='triclinic',
    description="Box type:"
)

display(solvent_widget, ff_widget, buffer_widget, ph_widget, box_type_widget)

# Protonation and Structure Preparation
A new PDB file is created from the selected chains and protonation is applied based on the chosen pH.
The output file is prepared for simulation.

In [ ]:
selected_chains = [
    chain_list[i]
    for i, cb in enumerate(chain_checkboxes)
    if cb.value
]

chain_string = "".join(selected_chains)
file_prefix = f"{base_name}_{chain_string}"

selected_chain_file = f"{file_prefix}.pdb"

with open(pdb_file) as infile, open(selected_chain_file, 'w') as outfile:
    for line in infile:
        if line.startswith(("ATOM", "HETATM")):
            cid = line[21].strip() or "A"
            if cid in selected_chains:
                outfile.write(line)
        else:
            outfile.write(line)

print("Chain file:", selected_chain_file)

output_pdb_pqr = f"{file_prefix}_processed.pdb"

!pdb2pqr --ff=AMBER \
         --titration-state-method=propka \
         --with-ph={ph_widget.value} \
         --drop-water \
         {gmx_input_pdb} \
         {output_pdb_pqr}

print("pdb2pqr output:", output_pdb_pqr)

# Generate GROMACS Input Files
Converts the processed PDB file into GROMACS format and generates topology using the selected force field.
This step prepares the required files for simulation.

In [ ]:
gmx_output = f"{file_prefix}_gmx_processed.pdb"

!gmx pdb2gmx \
  -f {output_pdb_pqr} \
  -o {gmx_output} \
  -water {solvent_widget.value} \
  -ff {ff_widget.value} \
  -ignh

print("pdb2gmx completed")

# Create Simulation Box
Generates a simulation box around the system using the selected box type and buffer distance.
This ensures enough space around the molecule.

👉 The buffer value defines the distance between the molecule and the box edges.

⚠️ Make sure to select the appropriate options below before running the cell. Selecting a protein structure is recommended for this step.

In [ ]:
gmx_output_editconf = f"{file_prefix}_newbox.pdb"

!gmx editconf \
    -f {gmx_output} \
    -o {gmx_output_editconf} \
    -bt {box_type_widget.value} \
    -c \
    -d {buffer_widget.value} \
    -princ

print("Box created")

# Solvate the System
Fills the simulation box with water molecules.
This step creates a realistic environment for the simulation.

👉 The system is now placed in a solvent environment.

In [ ]:
solvated_output = f"{file_prefix}_solv.pdb"

!gmx solvate \
    -cp {gmx_output_editconf} \
    -cs spc216.gro \
    -o {solvated_output} \
    -p topol.top

print("Solvation completed")

# Energy Minimization Parameters
Defines parameters for energy minimization before adding ions.
This step helps stabilize the system.

👉 This step runs automatically; no user input is required.

In [ ]:
%%writefile ions.mdp
integrator  = steep
emtol       = 1000.0
emstep      = 0.01
nsteps      = 500

# Prepare for Ion Addition
Prepares the system for ion addition and generates the required input file.
This step runs automatically.

👉 The system is checked and prepared for ion addition.

In [ ]:
!gmx grompp -f ions.mdp -c {solvated_output} -p topol.top -o ions.tpr -maxwarn 2

# Select Ion Types
Choose the positive and negative ions to be added to the system.
Default options are suitable for most cases.

👉 Ions are used to neutralize the system.

In [ ]:
import ipywidgets as widgets
from IPython.display import display

positive_ion_widget = widgets.Dropdown(
    options=['NA', 'K', 'LI', 'CA', 'MG'],
    value='NA',
    description="Positive Ion:"
)

negative_ion_widget = widgets.Dropdown(
    options=['CL', 'BR', 'I'],
    value='CL',
    description="Negative Ion:"
)

display(positive_ion_widget, negative_ion_widget)

# Add Ions and Neutralize the System
Selected ions are added to the system to achieve charge neutrality.
The final structure is now ready for molecular dynamics simulation.

👉 The system is now fully prepared for simulation.

In [ ]:
!echo SOL | gmx genion \
    -s ions.tpr \
    -o {file_prefix}_solv_ions.pdb \
    -p topol.top \
    -pname {positive_ion_widget.value} \
    -nname {negative_ion_widget.value} \
    -neutral \
    -conc 0.15

# Analyze and Relabel Chains
Chains in the final PDB file are analyzed and selected chains are relabeled.
This ensures a consistent and organized structure.

👉 Chain IDs are standardized (A, B, C ...)

In [ ]:
from collections import Counter

final_pdb = f"{file_prefix}_solv_ions.pdb"

chain_counts = Counter()

with open(final_pdb) as f:
    for line in f:
        if line.startswith("ATOM"):
            chain = line[21].strip() or "EMPTY"
            chain_counts[chain] += 1

protein_chains = [c for c in chain_counts if c != "EMPTY"]

ordered_selected_chains_from_file = [
    chain for chain in protein_chains if chain in selected_chains
]

new_labels = list("ABCDEFGHIJKLMNOPQRSTUVWXYZ")

chain_mapping = {}

for i, orig_chain_from_file in enumerate(ordered_selected_chains_from_file):
    if i < len(new_labels):
        chain_mapping[orig_chain_from_file] = new_labels[i]

print("Chain counts:", chain_counts)
print("Selected chains (user input):", selected_chains)
print("Protein chains (all protein chains from file):", protein_chains)
print("Ordered selected chains (based on file order):", ordered_selected_chains_from_file)
print("Mapping:", chain_mapping)

# Visualize the Final Structure
Explore the prepared system in 3D.
Chains are shown in different colors, while water and ions are displayed separately.
Hover over atoms to see detailed information.

👉 Rotate, zoom, and explore using your mouse.

In [ ]:
import py3Dmol
from itertools import cycle
from collections import Counter
import os

base_name_clean = os.path.splitext(os.path.basename(pdb_file))[0]
file_prefix = f"{base_name_clean}_{''.join(selected_chains)}"
final_pdb = f"{file_prefix}_solv_ions.pdb"

print("Using file:", final_pdb)

chain_counts = Counter()

with open(final_pdb) as f:
    for line in f:
        if line.startswith("ATOM"):
            chain = line[21].strip() or "EMPTY"
            chain_counts[chain] += 1

protein_chains = [c for c in chain_counts if c != "EMPTY"]

protein_chains_sorted = sorted(
    protein_chains,
    key=lambda x: chain_counts[x],
    reverse=True
)

updated_chains = protein_chains_sorted[:len(selected_chains)]
chain_mapping = dict(zip(selected_chains, updated_chains))

print("Mapping:", chain_mapping)

view = py3Dmol.view(width=900, height=650)

with open(final_pdb, 'r') as f:
    view.addModel(f.read(), 'pdb')

view.setStyle({}, {'cartoon': {'color': 'white'}})

colors = ['red','blue','green','orange','purple','cyan']
for orig, color in zip(selected_chains, cycle(colors)):
    new = chain_mapping.get(orig)
    if new:
        view.addStyle({'chain': new}, {'cartoon': {'color': color}})

view.addStyle({'resn':'SOL'}, {'sphere':{'radius':0.12,'opacity':0.2}})

view.addStyle({'resn': positive_ion_widget.value}, {'sphere':{'color':'orange'}})
view.addStyle({'resn': negative_ion_widget.value}, {'sphere':{'color':'green'}})

view.setHoverable(
    {},
    True,
    """
    function(atom,viewer,event,container) {
        if(!atom.label) {
            atom.label = viewer.addLabel(
                "Atom: " + atom.elem +
                "\\nResidue: " + atom.resn +
                "\\nResID: " + atom.resi +
                "\\nChain: " + atom.chain,
                {
                    position: atom,
                    backgroundColor: 'white',
                    fontColor: 'black',
                    borderThickness: 1,
                    fontSize: 12
                }
            );
        }
    }
    """,
    """
    function(atom,viewer) {
        if(atom.label) {
            viewer.removeLabel(atom.label);
            delete atom.label;
        }
    }
    """
)
view.zoomTo()
view.show()

# Download Results and Clean Workspace
All generated GROMACS files are packaged into a single zip archive for download.
After downloading, the workspace is cleaned for a fresh start.

👉 After downloading, you can restart the notebook for a new run.

In [ ]:
from google.colab import files
import os
import glob

zip_filename = "gromacs_preparation_files.zip"

files_to_zip = [
    final_pdb,
    "topol.top"
]

for f in glob.glob('topol_Protein*.itp'):
    files_to_zip.append(f)

for f in glob.glob('posre_Protein*.itp'):
    files_to_zip.append(f)

existing_files = [f for f in files_to_zip if os.path.exists(f)]

zip_command = f"zip -r {zip_filename} {' '.join(existing_files)}"
!{zip_command}

files.download(zip_filename)

print(f"{zip_filename} archive created and ready for download.")

print("Cleaning: removing all generated GROMACS files...")

files_to_delete = [
    pdb_file, # Corrected: Changed 'initial_pdb' to 'pdb_file'
    visual_pdb,
    gmx_input_pdb,
    selected_chain_file,
    output_pdb_pqr,
    gmx_output,
    gmx_output_editconf,
    solvated_output,
    final_pdb,
    solvated_output,
    "topol.top",
    "ions.mdp",
    "ions.tpr",
    f"{file_prefix}_processed.log",
    "mdout.mdp"
]

files_to_delete.extend(glob.glob('topol_Protein*.itp'))
files_to_delete.extend(glob.glob('posre_Protein*.itp'))
files_to_delete.extend(glob.glob('#*#'))

for f in set(files_to_delete):
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Deleted: {f}")
        except Exception as e:
            print(f"Could not delete {f}: {e}")

print("Cleaning completed. All GROMACS files have been removed.")